# Frozen DeBERTa + LightGBM
Attach the competition data and the exp2 input containing `boosters/` and `encoder/`. Use a GPU accelerator. The notebook runs offline and writes `submission.csv`.

In [ ]:
import json
import re
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "llm-classification-finetuning"
competition_paths = [
    INPUT_ROOT / "competitions" / COMPETITION,
    INPUT_ROOT / COMPETITION,
]
DATA_DIR = next((p for p in competition_paths if (p / "test.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Attach the LLM Classification Finetuning competition data.")

bundles = [p.parent.parent for p in INPUT_ROOT.rglob("boosters/config.json")
           if (p.parent.parent / "encoder" / "config.json").is_file()]
if len(bundles) != 1:
    raise ValueError("Attach one exp2 input with the boosters/ and encoder/ directories.")
BUNDLE = bundles[0]
config = json.loads((BUNDLE / "boosters" / "config.json").read_text())
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8
max_length = config["max_length"]

In [ ]:
TEXT_COLS = ["prompt", "response_a", "response_b"]
EMPTY_TURN = "[EMPTY]"


def parse_turns(x: str) -> list[str]:
    """Parse a JSON-serialized list of turns (i-th item = i-th dialog turn).

    The raw columns are JSON, not Python literals: json.loads handles `null`
    and JSON-only escapes that literal_eval chokes on. None / blank turns are
    replaced with EMPTY_TURN so rows are kept. Raises if the value is not a
    JSON list: the data is known to parse fully, so a failure should be loud.
    """
    parsed = json.loads(x)
    if not isinstance(parsed, list):
        raise ValueError(f"expected a JSON list, got {type(parsed).__name__}: {x[:100]!r}")
    return [t if isinstance(t, str) and t.strip() else EMPTY_TURN for t in parsed]



def sanitize_text(text: str) -> str:
    return text.encode("utf-8", errors="surrogatepass").decode("utf-8", errors="replace")


def parse_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    parsed = {col: df[col].apply(parse_turns) for col in TEXT_COLS}

    n_turns = parsed["prompt"].apply(len)
    for col in ("response_a", "response_b"):
        assert (parsed[col].apply(len) == n_turns).all(), \
            f"{col}: number of turns differs from prompt"

    for col in TEXT_COLS:
        n_empty = parsed[col].apply(lambda turns: EMPTY_TURN in turns).sum()
        print(f"[parse] {col}: {n_empty} rows with an empty turn replaced by {EMPTY_TURN}")
        df[f"{col}_text"] = parsed[col].apply(
            lambda turns: sanitize_text(" ".join(turns).strip())
        )
    return df

In [ ]:
REL_LEN_CLIP = 50

REFUSAL_PATTERN = re.compile(
    r"\b(?:i cannot|i can't|i'm sorry|i am sorry|as an ai|i'm not able to|"
    r"i don't have the ability|i'm unable to|cannot provide|can't provide)\b",
    flags=re.IGNORECASE,
)


def add_length_features(df: pd.DataFrame) -> pd.DataFrame:
    df["prompt_len_chars"] = df["prompt_text"].str.len()
    df["response_a_len_chars"] = df["response_a_text"].str.len()
    df["response_b_len_chars"] = df["response_b_text"].str.len()
    df["len_diff"] = df["response_a_len_chars"] - df["response_b_len_chars"]

    df["a_rel_len"] = df["response_a_len_chars"] / (df["prompt_len_chars"] + 1)
    df["b_rel_len"] = df["response_b_len_chars"] / (df["prompt_len_chars"] + 1)
    rel_len_diff = df["a_rel_len"] - df["b_rel_len"]
    df["rel_len_diff"] = rel_len_diff.clip(-REL_LEN_CLIP, REL_LEN_CLIP)
    return df


def count_markdown_features(text: str) -> dict:
    return {
        "n_bullets": len(re.findall(r"(?:^|\n)[\-\*]\s", text)),
        "n_numbered": len(re.findall(r"(?:^|\n)\d+\.\s", text)),
        "n_headers": len(re.findall(r"(?:^|\n)#{1,6}\s", text)),
        "n_bold": len(re.findall(r"\*\*[^*]+\*\*", text)),
        "n_code_blocks": text.count("```"),
    }


def add_formatting_features(df: pd.DataFrame) -> pd.DataFrame:
    feat_a = df["response_a_text"].apply(count_markdown_features).apply(pd.Series).add_prefix("a_")
    feat_b = df["response_b_text"].apply(count_markdown_features).apply(pd.Series).add_prefix("b_")
    df = pd.concat([df, feat_a, feat_b], axis=1)

    for feat in ["n_bullets", "n_numbered", "n_headers", "n_bold", "n_code_blocks"]:
        df[f"{feat}_diff"] = df[f"a_{feat}"] - df[f"b_{feat}"]
    return df


def add_refusal_features(df: pd.DataFrame) -> pd.DataFrame:
    df["a_refusal"] = df["response_a_text"].str.contains(REFUSAL_PATTERN, regex=True).astype(int)
    df["b_refusal"] = df["response_b_text"].str.contains(REFUSAL_PATTERN, regex=True).astype(int)
    df["refusal_diff"] = df["a_refusal"] - df["b_refusal"]
    return df

In [ ]:
TEXT_COLUMNS = ["prompt_text", "response_a_text", "response_b_text"]

def encode_pairs(df, tokenizer, encoder):
    difference = np.empty((len(df), encoder.config.hidden_size), dtype=np.float32)
    similarity = np.empty((len(df), 2), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(df), batch_size):
            stop = min(start + batch_size, len(df))
            prompts, responses_a, responses_b = (
                df[column].iloc[start:stop].tolist() for column in TEXT_COLUMNS
            )
            batch = tokenizer(
                prompts + prompts,
                responses_a + responses_b,
                padding=True,
                truncation="longest_first",
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            hidden = encoder(**batch).last_hidden_state.float()
            mask = batch.attention_mask.unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            h_a, h_b = pooled.chunk(2)
            delta = h_a - h_b
            difference[start:stop] = delta.cpu().numpy()
            similarity[start:stop, 0] = F.cosine_similarity(h_a, h_b).cpu().numpy()
            similarity[start:stop, 1] = delta.norm(dim=1).cpu().numpy()
            if start % (100 * batch_size) == 0 or stop == len(df):
                print(f"Encoded {stop}/{len(df)} pairs", flush=True)
    return similarity, difference

In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
test = parse_text_columns(test)
test = add_refusal_features(add_formatting_features(add_length_features(test)))

tokenizer = AutoTokenizer.from_pretrained(BUNDLE / "encoder", local_files_only=True)
encoder = AutoModel.from_pretrained(BUNDLE / "encoder", local_files_only=True)
encoder = encoder.requires_grad_(False).eval().to(device)
similarity, difference = encode_pairs(test, tokenizer, encoder)
features = np.column_stack((test[config["feature_cols"]].to_numpy(), similarity, difference))
if not np.isfinite(features).all():
    raise ValueError("Test features contain non-finite values.")

In [ ]:
fold_predictions = []
for fold in config["folds"]:
    model = lgb.Booster(model_file=str(BUNDLE / "boosters" / f"fold{fold}.txt"))
    fold_predictions.append(model.predict(features, num_threads=1))

probabilities = np.mean(fold_predictions, axis=0)
columns = {"a": "winner_model_a", "b": "winner_model_b", "tie": "winner_tie"}
prediction = pd.DataFrame(probabilities, columns=[columns[c] for c in config["classes"]])
prediction.insert(0, "id", test["id"])
submission = prediction.set_index("id").reindex(sample["id"]).reset_index()[sample.columns]
values = submission.drop(columns="id").to_numpy()
if not np.isfinite(values).all() or not np.allclose(values.sum(axis=1), 1):
    raise ValueError("Submission has missing or invalid probabilities.")
submission.to_csv("submission.csv", index=False)
submission.head()